# 07 — Path Reconstruction (Dead Reckoning)

Integrate `GYR_Y_DEG` over time to reconstruct the scooter's approximate
heading trajectory. Plot relative path shape (no GPS required).

> **Note**: errors accumulate over time. Use for short segments or relative shape only.

In [1]:
from pathlib import Path
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from scooter_imu.data.loader import load_raw
from scooter_imu.data.preprocessor import preprocess
from scooter_imu.algorithms.path_reconstruction import PathReconstructor

RAW_PATH = Path("../data/raw/ScooterIMUData.csv")

In [2]:
df = preprocess(load_raw(RAW_PATH))
result = PathReconstructor(initial_heading_deg=0.0).predict(df)
result[["timestamp", "gyr_y", "heading_delta_deg", "heading_deg"]].head(10)

,timestamp,gyr_y,heading_delta_deg,heading_deg
0,2024-03-15 10:00:00.000,-0.001133,-0.001298,-0.001298
1,2024-03-15 10:00:00.020,-0.001249,-0.001431,-0.002730
2,2024-03-15 10:00:00.040,0.002650,0.003037,0.000307
3,2024-03-15 10:00:00.060,0.000661,0.000757,0.001065
4,2024-03-15 10:00:00.080,-0.000423,-0.000485,0.000580
5,2024-03-15 10:00:00.100,0.000997,0.001142,0.001722
6,2024-03-15 10:00:00.120,-0.004214,-0.004829,-0.003107
7,2024-03-15 10:00:00.140,-0.000086,-0.000099,-0.003205
8,2024-03-15 10:00:00.160,0.003995,0.004578,0.001373
9,2024-03-15 10:00:00.180,0.000265,0.000304,0.001676


In [3]:
fig = px.line(result, x="timestamp", y="heading_deg",
              title="Reconstructed Heading Over Time",
              labels={"heading_deg": "Heading (deg)"})
fig.show()

In [4]:
# Approximate 2-D path (unit speed assumed; shows shape only, not scale)
angles_rad = np.deg2rad(result["heading_deg"].to_numpy())
x = np.cumsum(np.cos(angles_rad) * result["dt"].to_numpy())
y = np.cumsum(np.sin(angles_rad) * result["dt"].to_numpy())

fig = go.Figure(go.Scatter(x=x, y=y, mode="lines",
                           line=dict(color="royalblue", width=2)))
fig.add_trace(go.Scatter(x=[x[0]], y=[y[0]], mode="markers",
                         name="Start", marker=dict(color="green", size=12)))
fig.add_trace(go.Scatter(x=[x[-1]], y=[y[-1]], mode="markers",
                         name="End", marker=dict(color="red", size=12)))
fig.update_yaxes(scaleanchor="x")
fig.update_layout(title="Approximate Path Shape (Dead Reckoning)",
                  xaxis_title="X (relative)", yaxis_title="Y (relative)")
fig.show()

## Animated Scooter on Road

Watch the 🛵 travel its reconstructed path in real time.
Trail segments are coloured by turning intensity — blue = straight, orange/red = sharp turn.
Use the **Play** button or drag the slider to scrub through the trip.

In [5]:
import numpy as np
import plotly.graph_objects as go

angles_rad = np.deg2rad(result["heading_deg"].to_numpy())
x_path = np.cumsum(np.cos(angles_rad) * result["dt"].to_numpy())
y_path = np.cumsum(np.sin(angles_rad) * result["dt"].to_numpy())
turn_intensity = np.abs(result["gyr_y"].to_numpy())

pad = max(abs(x_path).max(), abs(y_path).max()) * 0.15
x_range = [x_path.min() - pad, x_path.max() + pad]
y_range = [y_path.min() - pad, y_path.max() + pad]

n_frames = min(200, len(x_path))
idx = np.linspace(0, len(x_path) - 1, n_frames, dtype=int)

def trail_color(intensity):
    t = min(intensity / 1.0, 1.0)
    r = int(30 + 220 * t)
    g = int(100 * (1 - t))
    b = int(200 * (1 - t))
    return f"rgb({r},{g},{b})"

frames = []
for k, i in enumerate(idx):
    seg_colours = [trail_color(turn_intensity[j]) for j in idx[: k + 1]]
    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=x_path[idx[: k + 1]],
                    y=y_path[idx[: k + 1]],
                    mode="lines+markers",
                    line=dict(color="royalblue", width=2),
                    marker=dict(color=seg_colours, size=4),
                    showlegend=False,
                ),
                go.Scatter(
                    x=[x_path[i]],
                    y=[y_path[i]],
                    mode="text",
                    text=["🛵"],
                    textfont=dict(size=24),
                    showlegend=False,
                ),
            ],
            name=str(k),
        )
    )

waypoints = go.Scatter(
    x=[x_path[0], x_path[-1]],
    y=[y_path[0], y_path[-1]],
    mode="markers+text",
    marker=dict(color=["#00cc55", "#cc2200"], size=14, symbol="circle"),
    text=["  Start", "  End"],
    textposition="middle right",
    name="Waypoints",
)

fig = go.Figure(
    data=[
        go.Scatter(x=[x_path[0]], y=[y_path[0]], mode="lines",
                   line=dict(color="royalblue", width=2), showlegend=False),
        go.Scatter(x=[x_path[0]], y=[y_path[0]], mode="text",
                   text=["🛵"], textfont=dict(size=24), showlegend=False),
        waypoints,
    ],
    layout=go.Layout(
        title="🛵  Animated Scooter Path — colour = turning intensity",
        xaxis=dict(title="X (relative)", range=x_range, showgrid=True,
                   gridcolor="#dde", zeroline=False),
        yaxis=dict(title="Y (relative)", range=y_range, scaleanchor="x",
                   showgrid=True, gridcolor="#dde", zeroline=False),
        paper_bgcolor="#f4f6fb",
        plot_bgcolor="#e8eef8",
        height=580,
        showlegend=True,
        updatemenus=[dict(
            type="buttons", showactive=False, y=-0.14, x=0.5, xanchor="center",
            buttons=[
                dict(label="▶  Play", method="animate",
                     args=[None, dict(frame=dict(duration=40, redraw=True), fromcurrent=True)]),
                dict(label="⏸  Pause", method="animate",
                     args=[[None], dict(frame=dict(duration=0), mode="immediate")]),
            ],
        )],
        sliders=[dict(
            steps=[dict(args=[[f.name], dict(frame=dict(duration=0), mode="immediate")],
                        method="animate", label="") for f in frames],
            active=0, y=-0.06, len=0.92, x=0.04,
            currentvalue=dict(visible=False),
        )],
    ),
    frames=frames,
)
fig.show()